# Part 1: Data Cleaning
In this section, we load the dataset, identify missing values, handle them, and remove outliers and duplicates.

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('../data/train.csv')

# Identify missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Impute Age with median
df['Age'] = df['Age'].fillna(df['Age'].median())

# Impute Fare with median
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# Create Cabin indicator (1 if Cabin is known, 0 otherwise)
df['HasCabin'] = df['Cabin'].notna().astype(int)
df = df.drop(columns=['Cabin']) # Drop original Cabin column as it has too many missing values

# Impute Embarked with the mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Outlier handling for Fare (Cap at 99th percentile)
fare_99th = df['Fare'].quantile(0.99)
df['Fare'] = np.where(df['Fare'] > fare_99th, fare_99th, df['Fare'])

# Standardize Sex values just in case
df['Sex'] = df['Sex'].str.lower().str.strip()

# Remove duplicates
df = df.drop_duplicates()

print("\nMissing values after cleaning:")
print(df.isnull().sum())

# Save cleaned dataset
df.to_csv('../data/train_cleaned.csv', index=False)

# Part 2: Feature Engineering
We will engineer new features, encode categorical variables, and apply transformations to skewed features.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Feature Engineering
# Create FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Create IsAlone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
# Group rare titles
rare_titles = ['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# Fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

# Categorical Encoding
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

# Log transform Fare
df['LogFare'] = np.log1p(df['Fare'])

# Optional: Visualize LogFare
plt.figure()
sns.histplot(df['LogFare'], kde=True)
plt.title('Log Transformed Fare Distribution')
plt.show()

# Drop unnecessary columns
df = df.drop(columns=['Name', 'Ticket'])

# Part 3: Feature Selection
Now we perform correlation analysis and use a Random Forest to rank feature importance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Correlation Analysis
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Prepare features and target
X = df.drop(columns=['Survived', 'PassengerId'])
y = df['Survived']

# Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = rf.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

print("\nFeature Importances:")
print(importance_df)

# Final Selected Features
selected_features = importance_df.head(10)['Feature'].tolist()
print("\nTop 10 Selected Features:", selected_features)